# pgl Postgres installation — macOS pilot

This notebook installs and configures a local PostgreSQL database for the
Session Data Store pilot.

Run the cells from top to bottom.

## Requirements

- macOS, on Apple Silicon or Intel.
- A working Python 3.10+ Jupyter kernel running on this Mac.
- Internet access for installation.
- Homebrew installed on this Mac.

If Homebrew is not installed, run this command in **macOS Terminal**:

    /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

Follow the installer's instructions before continuing.

## Important

- Run Jupyter as your normal macOS user, not as root.
- Passwords are requested privately and are not printed.
- Do not put passwords in notebook cells or commit them to Git.
- Database files live outside this notebook and survive kernel restarts.
- This notebook starts Postgres but does not configure automatic startup after reboot.
- This is a macOS installer; Linux support can be added separately.
- This notebook does not create the scientific tables. Those will be created
  through numbered database migrations.

In [1]:
# ======================== CONFIGURATION ========================

POSTGRES_VERSION = "17"
POSTGRES_HOST = "127.0.0.1"  # Local-only networking.
POSTGRES_PORT = 5432

DATABASE_NAME = "pgl"
DATABASE_SCHEMA = "pgl"
DATABASE_USER = "pgl_app"
ADMIN_USER = "pgl_admin"

INSTALL_DIRECTORY = "~/.local/share/pgl/postgres"

INSTALL_POSTGRES_IF_MISSING = True
STARTUP_TIMEOUT_SECONDS = 60
CONNECT_TIMEOUT_SECONDS = 10

# If these environment variables are absent, the notebook prompts privately.
ADMIN_PASSWORD_ENV = "PGL_POSTGRES_ADMIN_PASSWORD"
DATABASE_PASSWORD_ENV = "PGL_POSTGRES_PASSWORD"

# Existing application passwords are preserved unless explicitly requested.
RESET_APPLICATION_PASSWORD = False

# ===============================================================

## Install the Python database driver

This installs Psycopg into the current notebook kernel's environment.

PostgreSQL itself is installed separately in the next steps.

In [2]:
%pip install "psycopg[binary]>=3.2,<4"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import getpass
import os
import platform
import re
import shlex
import shutil
import socket
import subprocess
import tempfile
from pathlib import Path

import psycopg
from psycopg import sql


def run_command(arguments, capture=False):
    """Run a command without invoking a shell."""
    return subprocess.run([str(argument) for argument in arguments], check=True, text=True, stdout=subprocess.PIPE if capture else None, env=COMMAND_ENV)


def ask_password(environment_name, prompt, confirm=False):
    """Read a password from the environment or request it privately."""
    password = os.environ.get(environment_name)

    if password is None:
        password = getpass.getpass(prompt)
        if confirm and password != getpass.getpass("Confirm password: "):
            raise ValueError("Passwords do not match.")

    if not password or any(character in password for character in ("\n", "\r", "\x00")):
        raise ValueError("Passwords must be nonempty and contain no newline or NUL characters.")

    return password


if platform.system() != "Darwin":
    raise RuntimeError("This notebook currently supports macOS only.")

if os.geteuid() == 0:
    raise RuntimeError("Run Jupyter as your normal macOS user, not as root.")

if POSTGRES_HOST != "127.0.0.1":
    raise ValueError("This pilot installer only supports local connections through 127.0.0.1.")

if not isinstance(POSTGRES_PORT, int) or not 1 <= POSTGRES_PORT <= 65535:
    raise ValueError("POSTGRES_PORT must be an integer between 1 and 65535.")

if not re.fullmatch(r"[0-9]+", POSTGRES_VERSION):
    raise ValueError("POSTGRES_VERSION must be a major version number such as '17'.")

for name in (DATABASE_NAME, DATABASE_SCHEMA, DATABASE_USER, ADMIN_USER):
    if not name or "\x00" in name or len(name.encode("utf-8")) > 63:
        raise ValueError("Database, schema, and role names must contain 1–63 UTF-8 bytes and no NUL.")

if DATABASE_USER == ADMIN_USER:
    raise ValueError("Use different administrator and application accounts.")

if DATABASE_USER.startswith("pg_") or ADMIN_USER.startswith("pg_"):
    raise ValueError("Postgres reserves role names beginning with pg_.")

if DATABASE_SCHEMA.startswith("pg_") or DATABASE_SCHEMA in {"public", "information_schema"}:
    raise ValueError("Choose a dedicated application schema.")

if DATABASE_NAME in {"postgres", "template0", "template1"}:
    raise ValueError("Choose a dedicated application database.")

brew_candidates = [shutil.which("brew"), "/opt/homebrew/bin/brew", "/usr/local/bin/brew"]
BREW = next((Path(path) for path in brew_candidates if path and Path(path).is_file()), None)

if BREW is None:
    raise RuntimeError("Homebrew was not found. Install it in macOS Terminal using the instructions above, then rerun this cell.")

COMMAND_ENV = os.environ.copy()
COMMAND_ENV["PATH"] = f"{BREW.parent}:{COMMAND_ENV.get('PATH', '')}"

ROOT_DIRECTORY = Path(INSTALL_DIRECTORY).expanduser().resolve()
DATA_DIRECTORY = ROOT_DIRECTORY / "data"
LOG_FILE = ROOT_DIRECTORY / "postgres.log"
FORMULA = f"postgresql@{POSTGRES_VERSION}"

print(f"Homebrew: {BREW}")
print(f"Database directory: {DATA_DIRECTORY}")
print(f"Local endpoint: {POSTGRES_HOST}:{POSTGRES_PORT}")

Homebrew: /opt/homebrew/bin/brew
Database directory: /Users/justin/.local/share/pgl/postgres/data
Local endpoint: 127.0.0.1:5432


## Install PostgreSQL

This installs PostgreSQL binaries if they are missing.

Homebrew may also initialize its own default database directory. This notebook
does not use that directory and does not start the Homebrew-managed service.

Our database will use the dedicated directory configured above.

In [4]:
BREW_PREFIX = Path(run_command([BREW, "--prefix"], capture=True).stdout.strip())
BINARY_DIRECTORY = BREW_PREFIX / "opt" / FORMULA / "bin"

if not (BINARY_DIRECTORY / "postgres").is_file():
    if not INSTALL_POSTGRES_IF_MISSING:
        raise RuntimeError(f"{FORMULA} is missing and installation is disabled.")
    run_command([BREW, "install", FORMULA])

INITDB = BINARY_DIRECTORY / "initdb"
PG_CTL = BINARY_DIRECTORY / "pg_ctl"

for executable in (INITDB, PG_CTL, BINARY_DIRECTORY / "postgres"):
    if not executable.is_file():
        raise RuntimeError(f"Expected executable is missing: {executable}")

run_command([BINARY_DIRECTORY / "postgres", "--version"])

==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> Downloading https://ghcr.io/v2/homebrew/core/portable-ruby/blobs/sha256:e0088dff5614b39387300136ec7a5f95bf1e07589547245c919524fc9e8b4197
######################################################################## 100.0%
==> Pouring portable-ruby-4.0.7.arm64_big_sur.bottle.tar.gz
==> Downloading Homebrew API data
✔︎ JSON API packages.arm64_tahoe.jws.json
==> Auto-updated Homebrew!
Updated 2 taps (homebrew/core and homebrew/cask).
==> New Formulae
actions-languageserver: Language server for GitHub Actions YAML files
adrs: Architectural Decision Record tool in Rust
advai-cli: CLI for managing skills and external CLIs
agent-manager: Run Claude Code, Codex, OpenCode and other AI coding agents in tmux
ai-cli: Generate images, video, audio, and text from the terminal
all-smi: GPU monitorin

==> Would install 1 formula:
postgresql@17 17.11
==> Would install 1 dependency for postgresql@17:
json-c
==> Would upgrade 4 dependencies for postgresql@17:
ca-certificates
openssl@3
readline
xz


==> Fetching downloads for: postgresql@17
✔︎ Bottle Manifest ca-certificates (2026-09-25)
✔︎ Bottle Manifest readline (8.3.6)
✔︎ Bottle Manifest json-c (0.19)
✔︎ Bottle Manifest openssl@3 (3.6.5)
✔︎ Bottle Manifest xz (5.8.4)
✔︎ Bottle json-c (0.19)
✔︎ Bottle ca-certificates (2026-09-25)
✔︎ Bottle readline (8.3.6)
✔︎ Bottle xz (5.8.4)
✔︎ Bottle openssl@3 (3.6.5)
✔︎ Bottle postgresql@17 (17.11)


==> Installing dependencies for postgresql@17: ca-certificates, openssl@3 and json-c
==> Upgrading postgresql@17 dependency: ca-certificates
==> Pouring ca-certificates--2026-09-25.all.bottle.tar.gz
🍺  /opt/homebrew/Cellar/ca-certificates/2026-09-25: 5 files, 204.4KB
==> Upgrading postgresql@17 dependency: openssl@3
==> Pouring openssl@3--3.6.5.arm64_tahoe.bottle.tar.gz
🍺  /opt/homebrew/Cellar/openssl@3/3.6.5: 7,642 files, 37.8MB
==> Installing postgresql@17 dependency: json-c
==> Pouring json-c--0.19.arm64_tahoe.bottle.1.tar.gz
🍺  /opt/homebrew/Cellar/json-c/0.19: 34 files, 362KB
==> Installing postgresql@17
==> Pouring postgresql@17--17.11.arm64_tahoe.bottle.tar.gz
The files belonging to this database system will be owned by user "justin".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

fixing permissions on existing dire

CompletedProcess(args=['/opt/homebrew/opt/postgresql@17/bin/postgres', '--version'], returncode=0)

## Initialize the dedicated database directory

On the first run, choose an administrator password and save it in your password
manager.

On later runs, enter the existing administrator password.

This cell does not reset administrator passwords or replace existing database
files. A different Postgres major version requires an explicit upgrade.

In [6]:
ROOT_DIRECTORY.mkdir(parents=True, exist_ok=True, mode=0o700)

VERSION_FILE = DATA_DIRECTORY / "PG_VERSION"
NEW_CLUSTER = not VERSION_FILE.exists()

if not NEW_CLUSTER and VERSION_FILE.read_text().strip() != POSTGRES_VERSION:
    raise RuntimeError("Existing database files use another Postgres major version. Stop here and plan an explicit upgrade.")

if NEW_CLUSTER and DATA_DIRECTORY.exists() and any(DATA_DIRECTORY.iterdir()):
    raise RuntimeError(f"Refusing to initialize a nonempty directory: {DATA_DIRECTORY}")

admin_password = ask_password(ADMIN_PASSWORD_ENV, "Postgres administrator password: ", confirm=NEW_CLUSTER)

if NEW_CLUSTER:
    # Temporary password file is private and removed after initialization.
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8") as password_file:
        password_file.write(admin_password + "\n")
        password_file.flush()
        run_command([INITDB, "-D", DATA_DIRECTORY, "-U", ADMIN_USER, "--encoding=UTF8", "--auth=scram-sha-256", f"--pwfile={password_file.name}"])

    print("Initialized a new PostgreSQL database directory.")
else:
    print("Using the existing PostgreSQL database directory.")

The files belonging to this database system will be owned by user "justin".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

creating directory /Users/justin/.local/share/pgl/postgres/data ... ok
creating subdirectories ... ok
selecting dynamic shared memory implementation ... posix
selecting default "max_connections" ... 100
selecting default "shared_buffers" ... 128MB
selecting default time zone ... America/Los_Angeles
creating configuration files ... ok
running bootstrap script ... ok
performing post-bootstrap initialization ... ok
syncing data to disk ... ok

Success. You can now start the database server using:

    '/opt/homebrew/opt/postgresql@17/bin/pg_ctl' -D /Users/justin/.local/share/pgl/postgres/data -l logfile start

Initialized a new PostgreSQL database directory.


## Start Postgres

Postgres will listen only on `127.0.0.1`. Unix-domain sockets are disabled for
this dedicated instance.

If another server is already using the configured port, this cell stops rather
than silently connecting to it. Choose another port, such as 5433, in the
configuration cell and rerun the relevant cells.

If this instance is already running, it is left running. Changing its port
requires stopping it before starting it again.

In [7]:
status = subprocess.run([str(PG_CTL), "-D", str(DATA_DIRECTORY), "status"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=COMMAND_ENV)

if status.returncode == 0:
    print("This database instance is already running.")
elif status.returncode == 3:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        try:
            probe.bind((POSTGRES_HOST, POSTGRES_PORT))
        except OSError as error:
            raise RuntimeError(f"Port {POSTGRES_PORT} is unavailable. Choose another POSTGRES_PORT.") from error

    server_options = shlex.join(["-h", POSTGRES_HOST, "-p", str(POSTGRES_PORT), "-k", ""])
    run_command([PG_CTL, "-D", DATA_DIRECTORY, "-l", LOG_FILE, "-o", server_options, "-t", str(STARTUP_TIMEOUT_SECONDS), "-w", "start"])
    print("Postgres started.")
else:
    raise RuntimeError(f"Could not inspect the database instance; pg_ctl exited with status {status.returncode}.")

print(f"Server log: {LOG_FILE}")

waiting for server to start.... done
server started
Postgres started.
Server log: /Users/justin/.local/share/pgl/postgres/postgres.log


In [8]:
def admin_connect(database="postgres"):
    return psycopg.connect(host=POSTGRES_HOST, port=POSTGRES_PORT, dbname=database, user=ADMIN_USER, password=admin_password, connect_timeout=CONNECT_TIMEOUT_SECONDS, autocommit=True)


with admin_connect() as connection:
    actual_directory = Path(connection.execute("SHOW data_directory").fetchone()[0]).resolve()
    actual_listen_addresses = connection.execute("SHOW listen_addresses").fetchone()[0]
    actual_socket_directory = connection.execute("SHOW unix_socket_directories").fetchone()[0]

    if actual_directory != DATA_DIRECTORY:
        raise RuntimeError(f"Connected to an unexpected database directory: {actual_directory}")

    if actual_listen_addresses != POSTGRES_HOST or actual_socket_directory != "":
        raise RuntimeError("The running server does not match this notebook's local-only networking configuration.")

    server_version = connection.execute("SHOW server_version").fetchone()[0]

print(f"Verified PostgreSQL {server_version}")
print(f"Verified database directory: {actual_directory}")
print("Verified local-only networking.")

Verified PostgreSQL 17.11 (Homebrew)
Verified database directory: /Users/justin/.local/share/pgl/postgres/data
Verified local-only networking.


## Create the application account and database

Python/pgl uses a non-superuser account, not the administrator account.

On the first run:
- Choose an application password.
- The notebook creates the application role and database.
- The application owns its dedicated schema and can create tables there.

On later runs:
- Enter the existing application password.
- Existing passwords are preserved unless `RESET_APPLICATION_PASSWORD = True`.
- Existing data is not deleted.

The password is needed again when connecting from a fresh Python process.
Save it in your password manager.

In [10]:
with admin_connect() as connection:
    existing_role = connection.execute("SELECT rolsuper, rolcreatedb, rolcreaterole, rolreplication, rolbypassrls, rolcanlogin FROM pg_roles WHERE rolname = %s", (DATABASE_USER,)).fetchone()

    if existing_role is not None:
        if any(existing_role[:5]) or not existing_role[5]:
            raise RuntimeError("The existing application role has unexpected privileges or cannot log in. Review it before continuing.")

    database_exists = connection.execute("SELECT EXISTS (SELECT FROM pg_database WHERE datname = %s)", (DATABASE_NAME,)).fetchone()[0]

setting_password = existing_role is None or RESET_APPLICATION_PASSWORD
password_prompt = "Choose the application password: " if setting_password else "Existing application password: "
application_password = ask_password(DATABASE_PASSWORD_ENV, password_prompt, confirm=setting_password)

with admin_connect() as connection:
    if existing_role is None:
        statement = sql.SQL("CREATE ROLE {} LOGIN NOSUPERUSER NOCREATEDB NOCREATEROLE NOREPLICATION NOBYPASSRLS PASSWORD {}").format(sql.Identifier(DATABASE_USER), sql.Literal(application_password))
        connection.execute(statement)
    elif RESET_APPLICATION_PASSWORD:
        statement = sql.SQL("ALTER ROLE {} PASSWORD {}").format(sql.Identifier(DATABASE_USER), sql.Literal(application_password))
        connection.execute(statement)

    if not database_exists:
        connection.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DATABASE_NAME)))

    connection.execute(sql.SQL("GRANT CONNECT ON DATABASE {} TO {}").format(sql.Identifier(DATABASE_NAME), sql.Identifier(DATABASE_USER)))

with admin_connect(DATABASE_NAME) as connection:
    schema_owner = connection.execute("SELECT pg_get_userbyid(nspowner) FROM pg_namespace WHERE nspname = %s", (DATABASE_SCHEMA,)).fetchone()

    if schema_owner is None:
        connection.execute(sql.SQL("CREATE SCHEMA {} AUTHORIZATION {}").format(sql.Identifier(DATABASE_SCHEMA), sql.Identifier(DATABASE_USER)))
    elif schema_owner[0] != DATABASE_USER:
        raise RuntimeError(f"The existing schema is owned by {schema_owner[0]}, not {DATABASE_USER}. Review ownership before continuing.")

    connection.execute(sql.SQL("ALTER ROLE {} IN DATABASE {} SET search_path = {}, public").format(sql.Identifier(DATABASE_USER), sql.Identifier(DATABASE_NAME), sql.Identifier(DATABASE_SCHEMA)))

print("Application account, database, and schema are ready.")

Application account, database, and schema are ready.


## Test the application connection

This verifies that Python can connect using the application account.

It also tests permission to create and write a table in the application schema.
The test runs inside a transaction that is rolled back, leaving no test table
or data behind.

In [11]:
import uuid

CONNECTION_SETTINGS = {"host": POSTGRES_HOST, "port": POSTGRES_PORT, "dbname": DATABASE_NAME, "user": DATABASE_USER, "connect_timeout": CONNECT_TIMEOUT_SECONDS}

test_table = sql.Identifier(DATABASE_SCHEMA, f"_pgl_install_test_{uuid.uuid4().hex}")

with psycopg.connect(**CONNECTION_SETTINGS, password=application_password) as connection:
    try:
        identity = connection.execute("SELECT current_database(), current_user, current_schema()").fetchone()

        if identity != (DATABASE_NAME, DATABASE_USER, DATABASE_SCHEMA):
            raise RuntimeError(f"Unexpected connection identity: {identity}")

        connection.execute(sql.SQL("CREATE TABLE {} (value integer NOT NULL)").format(test_table))
        connection.execute(sql.SQL("INSERT INTO {} (value) VALUES (%s)").format(test_table), (42,))
        result = connection.execute(sql.SQL("SELECT value FROM {}").format(test_table)).fetchone()[0]

        if result != 42:
            raise RuntimeError("Database read/write test failed.")
    finally:
        connection.rollback()

print(f"Database: {identity[0]}")
print(f"User: {identity[1]}")
print(f"Schema: {identity[2]}")
print("Python connection and schema read/write test passed.")
print("No test table was retained.")

Database: pgl
User: pgl_app
Schema: pgl
Python connection and schema read/write test passed.
No test table was retained.


## Installation complete

Postgres is running independently of this notebook. Closing Jupyter or
restarting its Python kernel does not stop Postgres.

After a Mac reboot, rerun the notebook to start the server again. Existing
database files are reused.

### Connection settings

Use `CONNECTION_SETTINGS` and the application password to connect from Python.
The settings deliberately exclude the password.

### Safety

- Do not delete the database directory: it contains the database.
- Do not run two copies of this installer concurrently against the same directory.
- Do not change the Postgres major version to upgrade an existing database.
- Database persistence is not a backup.
- Before storing important session records, add database backup and tested
  restore procedures.
- Scientific payload files will live separately and will also need backups.

### Sharing

Commit this notebook to the project repository without credentials.
Clear outputs before sharing if they include machine-specific information.

Running this notebook on another Mac creates an independent database there.
It does not connect that Mac to your existing database.

In [ ]:
# Uncomment to inspect the last 50 lines of the server log:
# print("\n".join(LOG_FILE.read_text(errors="replace").splitlines()[-50:]))

# Uncomment to stop this database instance cleanly:
# run_command([PG_CTL, "-D", DATA_DIRECTORY, "-m", "fast", "-t", str(STARTUP_TIMEOUT_SECONDS), "-w", "stop"])